<a href="https://colab.research.google.com/github/MouliAggarwal/Global-Earthquake-Analysis/blob/main/Earthquake_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving earthquake_data. 2.xlsx to earthquake_data. 2.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel("earthquake_data. 2.xlsx")
df.head()

,title,magnitude,date_time,cdi,mmi,alert,tsunami,sig,net,nst,dmin,gap,magType,depth,latitude,longitude,location,continent,country
0,"M 7.0 - 18 km SW of Malango, Solomon Islands",7.0,2022-11-22 02:03:00,8,7,green,1,768,us,117,0.509,17.0,mww,14.000,-9.7963,159.596,"Malango, Solomon Islands",Oceania,Solomon Islands
1,"M 6.9 - 204 km SW of Bengkulu, Indonesia",6.9,2022-11-18 13:37:00,4,4,green,0,735,us,99,2.229,34.0,mww,25.000,-4.9559,100.738,"Bengkulu, Indonesia",NaN,NaN
2,M 7.0 -,7.0,2022-11-12 07:09:00,3,3,green,1,755,us,147,3.125,18.0,mww,579.000,-20.0508,-178.346,NaN,Oceania,Fiji
3,"M 7.3 - 205 km ESE of Neiafu, Tonga",7.3,2022-11-11 10:48:00,5,5,green,1,833,us,149,1.865,21.0,mww,37.000,-19.2918,-172.129,"Neiafu, Tonga",NaN,NaN
4,M 6.6 -,6.6,2022-11-09 10:14:00,0,2,green,1,670,us,131,4.998,27.0,mww,624.464,-25.5948,178.278,NaN,NaN,NaN


In [ ]:

!pip install pycountry reverse_geocoder pycountry-convert

import pycountry
import reverse_geocoder as rg
import pycountry_convert as pc
import pandas as pd

In [ ]:
continent_dict = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica"
}

In [ ]:
def get_country_continent(latitude, longitude):
    """
    Determines the country and continent for given latitude and longitude coordinates.

    Args:
        latitude (float): The latitude coordinate.
        longitude (float): The longitude coordinate.

    Returns:
        pandas.Series: A Series containing the country name and continent name.
                       Returns (None, None) if coordinates are invalid or geocoding fails.
    """
    try:
        if pd.isna(latitude) or pd.isna(longitude):
            return pd.Series([None, None])

        result = rg.search((latitude, longitude))[0]
        country_code = result['cc']

        country_name = None
        try:
            country_obj = pycountry.countries.get(alpha_2=country_code)
            if country_obj:
                country_name = country_obj.name
        except KeyError:
            pass

        continent_name = None
        if country_code == 'TL':
            continent_name = 'Asia'
        else:
            try:
                # Convert Alpha-2 country code to continent code, then map to name
                continent_code = pc.country_alpha2_to_continent_code(country_code)
                continent_name = continent_dict.get(continent_code)
            except (KeyError, ValueError):
                # Handle cases where country_alpha2_to_continent_code fails
                pass

        return pd.Series([country_name, continent_name])

    except Exception as e:
        # Log any errors during geocoding and return None for both country and continent
        print(f"Error in get_country_continent for ({latitude}, {longitude}): {e}")
        return pd.Series([None, None])

In [ ]:
def guess_country(latitude, longitude):
    """
    Guesses the country based on latitude and longitude coordinates using `reverse_geocoder`.

    Args:
        latitude (float): The latitude coordinate.
        longitude (float): The longitude coordinate.

    Returns:
        str: The country name if found, otherwise None.
    """
    try:
        if pd.isna(latitude) or pd.isna(longitude):
            return None

        # Perform reverse geocoding
        result = rg.search((latitude, longitude))[0]
        country_code = result['cc']

        # Get the full country name using pycountry
        country_obj = pycountry.countries.get(alpha_2=country_code)
        if country_obj:
            return country_obj.name
        else:
            return None

    except Exception as e:
        print(f"Error guessing country for ({latitude}, {longitude}): {e}")
        return None

# Example usage of guess_country:
print("--- Example `guess_country` usage ---")
lat_example = -9.7963  # Example from Solomon Islands
lon_example = 159.596
guessed_country = guess_country(lat_example, lon_example)
print(f"For latitude {lat_example}, longitude {lon_example}, the guessed country is: {guessed_country}")

lat_usa = 34.0522 # Example from USA
lon_usa = -118.2437
guessed_country_usa = guess_country(lat_usa, lon_usa)
print(f"For latitude {lat_usa}, longitude {lon_usa}, the guessed country is: {guessed_country_usa}")
print("------------------------------------")

--- Example `guess_country` usage ---
For latitude -9.7963, longitude 159.596, the guessed country is: Solomon Islands
For latitude 34.0522, longitude -118.2437, the guessed country is: United States
------------------------------------


In [ ]:
# Apply the new function to the DataFrame to get country and continent from lat/lon
# We will only apply it to rows where 'continent' is still NaN and lat/lon are available

# Create new columns to store the results from reverse geocoding, initializing with object dtype
df['country_reverse_geo'] = pd.Series(dtype='object')
df['continent_reverse_geo'] = pd.Series(dtype='object')

# Filter rows where 'continent' is NaN and 'latitude', 'longitude' are not NaN
rows_to_process = df[df['continent'].isna() & df['latitude'].notna() & df['longitude'].notna()]

# Iterate and apply the function (can be slow, consider using df.apply in a more optimized way if dataset is huge)
# For better performance, we apply to a temporary DataFrame and then merge back

if not rows_to_process.empty:
    print(f"Processing {len(rows_to_process)} rows with missing continent using lat/lon...")
    results = rows_to_process.apply(lambda row: get_country_continent(row['latitude'], row['longitude']), axis=1)

    # Assign results back to the main DataFrame
    # Use .loc with .values to avoid potential SettingWithCopyWarning and ensure dtype consistency
    df.loc[rows_to_process.index, 'country_reverse_geo'] = results.iloc[:, 0].values
    df.loc[rows_to_process.index, 'continent_reverse_geo'] = results.iloc[:, 1].values

    # Update the 'continent' column with the newly found values where it was NaN
    df['continent'] = df['continent'].fillna(df['continent_reverse_geo'])

print("DataFrame after applying reverse geocoding with new function:")
display(df.head())

# Check the number of remaining NaN values in 'continent' column
print(f"Number of remaining NaN in 'continent' after reverse geocoding: {df['continent'].isna().sum()}")

# Display information about the 'continent' column again
print("\nInfo for 'continent' column:")
display(df['continent'].value_counts(dropna=False))

DataFrame after applying reverse geocoding with new function:


,title,magnitude,date_time,cdi,mmi,alert,tsunami,sig,net,nst,...,gap,magType,depth,latitude,longitude,location,continent,country,country_reverse_geo,continent_reverse_geo
0,"M 7.0 - 18 km SW of Malango, Solomon Islands",7.0,2022-11-22 02:03:00,8,7,green,1,768,us,117,...,17.0,mww,14.000,-9.7963,159.596,"Malango, Solomon Islands",Oceania,Solomon Islands,NaN,NaN
1,"M 6.9 - 204 km SW of Bengkulu, Indonesia",6.9,2022-11-18 13:37:00,4,4,green,0,735,us,99,...,34.0,mww,25.000,-4.9559,100.738,"Bengkulu, Indonesia",Asia,NaN,NaN,NaN
2,M 7.0 -,7.0,2022-11-12 07:09:00,3,3,green,1,755,us,147,...,18.0,mww,579.000,-20.0508,-178.346,NaN,Oceania,Fiji,NaN,NaN
3,"M 7.3 - 205 km ESE of Neiafu, Tonga",7.3,2022-11-11 10:48:00,5,5,green,1,833,us,149,...,21.0,mww,37.000,-19.2918,-172.129,"Neiafu, Tonga",Oceania,NaN,NaN,NaN
4,M 6.6 -,6.6,2022-11-09 10:14:00,0,2,green,1,670,us,131,...,27.0,mww,624.464,-25.5948,178.278,NaN,Oceania,NaN,NaN,NaN


Number of remaining NaN in 'continent' after reverse geocoding: 0

Info for 'continent' column:


,count
continent,
Asia,291
Oceania,242
North America,109
South America,105
Europe,25
Africa,7
Antarctica,3


In [ ]:
# Display a sample of rows where the continent is 'Oceania'
oceania_quakes = df[df['continent'] == 'Oceania'][['latitude', 'longitude', 'location', 'country', 'continent']]
display(oceania_quakes.head(10))

print(f"Total earthquakes assigned to Oceania: {len(oceania_quakes)}")

,latitude,longitude,location,country,continent
0,-9.7963,159.596,"Malango, Solomon Islands",Solomon Islands,Oceania
2,-20.0508,-178.346,NaN,Fiji,Oceania
3,-19.2918,-172.129,"Neiafu, Tonga",NaN,Oceania
4,-25.5948,178.278,NaN,NaN,Oceania
5,-26.0442,178.381,the Fiji Islands,NaN,Oceania
6,-25.9678,178.363,the Fiji Islands,NaN,Oceania
12,-21.2077,170.239,"Isangel, Vanuatu",NaN,Oceania
13,-6.2237,146.471,"Kainantu, Papua New Guinea",Papua New Guinea,Oceania
15,-32.6922,-178.959,the Kermadec Islands,NaN,Oceania
19,-54.1325,159.027,NaN,NaN,Oceania


Total earthquakes assigned to Oceania: 242


In [ ]:
# Apply the guess_country function to create a new column
df['country_from_lat_lon'] = df.apply(lambda row: guess_country(row['latitude'], row['longitude']), axis=1)

# Display the relevant columns to show the new country information
display(df[['latitude', 'longitude', 'location', 'country', 'country_from_lat_lon']].head(10))

# Check for any discrepancies between the original 'country' column and the new 'country_from_lat_lon'
discrepancies = df[df['country'].notna() & df['country_from_lat_lon'].notna() & (df['country'] != df['country_from_lat_lon'])]

if not discrepancies.empty:
    print("\nDiscrepancies found between 'country' and 'country_from_lat_lon' (showing first 5):")
    display(discrepancies[['latitude', 'longitude', 'location', 'country', 'country_from_lat_lon']].head())
else:
    print("\nNo major discrepancies found between 'country' and 'country_from_lat_lon' where both are present.")

,latitude,longitude,location,country,country_from_lat_lon
0,-9.7963,159.5960,"Malango, Solomon Islands",Solomon Islands,Solomon Islands
1,-4.9559,100.7380,"Bengkulu, Indonesia",NaN,Indonesia
2,-20.0508,-178.3460,NaN,Fiji,Tonga
3,-19.2918,-172.1290,"Neiafu, Tonga",NaN,Tonga
4,-25.5948,178.2780,NaN,NaN,Fiji
5,-26.0442,178.3810,the Fiji Islands,NaN,Fiji
6,-25.9678,178.3630,the Fiji Islands,NaN,Fiji
7,7.6712,-82.3396,"Boca Chica, Panama",Panama,Panama
8,18.3300,-102.9130,"Aguililla, Mexico",Mexico,Mexico
9,18.3667,-103.2520,"Aguililla, Mexico",Mexico,Mexico



Discrepancies found between 'country' and 'country_from_lat_lon' (showing first 5):


,latitude,longitude,location,country,country_from_lat_lon
2,-20.0508,-178.346,NaN,Fiji,Tonga
10,23.1444,121.307,"Yujing, Taiwan",Taiwan,"Taiwan, Province of China"
11,23.0290,121.348,"Lugu, Taiwan",Taiwan,"Taiwan, Province of China"
14,29.7263,102.279,"Kangding, China",People's Republic of China,China
24,23.3421,121.636,"Hualien City, Taiwan",Taiwan,"Taiwan, Province of China"


In [ ]:
# Save the modified DataFrame to a new Excel file
df.to_excel('earthquake_data_updated.xlsx', index=False)
print('DataFrame saved to earthquake_data_updated.xlsx')


DataFrame saved to earthquake_data_updated.xlsx


In [ ]:
from google.colab import files

# Initiate the download of the newly created Excel file to your local machine
files.download('earthquake_data_updated.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download('earthquake_data_updated.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>